# 🚀 Google Colab GPU - Optimized Islamic Knowledge Harvester

**Version:** 2.0 with GPU Manager

**Features:**
- ✅ Auto GPU detection (T4, P100, V100, K80)
- ✅ Auto batch size configuration
- ✅ Google Drive checkpoint storage
- ✅ Auto-checkpointing every N batches
- ✅ Manual resume from checkpoint
- ✅ Precision auto-tuning (float16/float32)
- ✅ Keyword pre-filtering (60% GPU reduction)
- ✅ Clip scoring (1-10)

---

## ⚠️ IMPORTANT: Before Running

1. **Enable GPU Runtime:**
   - Click `Runtime` → `Change runtime type`
   - Select `GPU` → `T4` (free tier)
   - Click `Save`

2. **Upload File:**
   - Upload `chunks.json` from your computer
   - File should be in `/content/` directory

3. **Processing Time:**
   - ~100 chunks: 1-2 minutes
   - ~500 chunks: 5-7 minutes
   - ~712 chunks: 7-10 minutes

---

## Step 1: Install Dependencies

In [ ]:
!pip install transformers accelerate sentencepiece torch --quiet
print("✓ Dependencies installed")

## Step 2: Initialize GPU Manager

In [ ]:
import sys
sys.path.append('/content')  # Ensure current directory is in path

from gpu_manager import GPUManager

# Initialize GPU manager
gpu_mgr = GPUManager()

# Detect GPU and get optimal config
gpu_available, gpu_type, config = gpu_mgr.detect_gpu()

# Mount Google Drive for checkpoints
drive_mounted = gpu_mgr.mount_google_drive()

print(f"\n✓ GPU Manager initialized")
print(f"  GPU Available: {gpu_available}")
print(f"  GPU Type: {gpu_type}")
print(f"  Batch Size: {config['batch_size']}")
print(f"  Precision: {config['precision']}")

## Step 3: Load Files & Check for Resume

In [ ]:
import json
import os

# Check for chunks.json
if 'chunks.json' not in os.listdir('/content/'):
    print("⚠️ chunks.json not found! Please upload it using the file icon on the left.")
else:
    with open('chunks.json') as f:
        all_chunks = json.load(f)
    print(f"✓ Loaded {len(all_chunks)} chunks from chunks.json")
    
    # Check for existing checkpoint
    checkpoints = gpu_mgr.list_checkpoints()
    if checkpoints:
        print(f"\n💾 Found {len(checkpoints)} checkpoint(s)")
        latest = checkpoints[0]
        print(f"  Latest: {latest.name}")

## Step 4: Load AI Model (Auto Precision)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Use smaller model for faster processing on free T4
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Get optimal precision from GPU manager
torch_dtype = gpu_mgr.get_precision()
print(f"Loading model: {MODEL_NAME}...")
print(f"Using precision: {torch_dtype}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    device_map="auto",
    low_cpu_mem_usage=True
)

print(f"✓ Model loaded on {'GPU' if gpu_available else 'CPU'}")

## Step 5: Define Optimized Processing Functions

In [ ]:
import re

# Valid Islamic topics
VALID_TOPICS = [
    "Tawheed (Oneness of Allah)",
    "Salah (Prayer)",
    "Zakat (Charity)",
    "Sawm (Fasting)",
    "Hajj (Pilgrimage)",
    "Akhlaq (Character/Manners)",
    "Sabr (Patience)",
    "Shukr (Gratitude)",
    "Love/Mercy",
    "Forgiveness",
    "Knowledge/Wisdom",
    "Death/Afterlife",
    "Dua (Supplication)",
    "Justice/Oppression",
    "Sin/Repentance",
    "Quran/Sunnah",
    "Other"
]

# IMPORTANT keywords for pre-filtering
IMPORTANT_KEYWORDS = [
    'charity', 'zakat', 'sadaqah', 'poor', 'needy', 'wealth', 'money', 'give',
    'oppression', 'tyrant', 'injustice', 'justice', 'fair', 'unfair', 'wrong',
    'prayer', 'salah', 'fasting', 'sawm', 'hajj', 'pilgrimage', 'quran',
    'patience', 'sabr', 'gratitude', 'shukr', 'mercy', 'rahmah', 'love', 'kindness',
    'hellfire', 'paradise', 'jannah', 'grave', 'death', 'judgment', 'day of',
    'faith', 'belief', 'trust', 'tawakkul', 'allah', 'god', 'prophet',
    'dua', 'supplication', 'pray', 'ask', 'forgive', 'repent'
]

def has_important_content(text):
    """Check if chunk contains important Islamic content."""
    text_lower = text.lower()
    for keyword in IMPORTANT_KEYWORDS:
        if keyword in text_lower:
            return True
    return False


def calculate_clip_score(text, topic, summary):
    """Calculate viral clip potential score (1-10)."""
    score = 5  # Base score
    text_lower = text.lower()
    
    # Emotional impact keywords (+2)
    emotional_words = ['love', 'mercy', 'fear', 'hope', 'paradise', 'hellfire', 
                       'beautiful', 'amazing', 'powerful', 'heart', 'soul']
    if any(word in text_lower for word in emotional_words):
        score += 2
    
    # Strong religious message (+1)
    religious_words = ['allah', 'prophet', 'quran', 'faith', 'belief', 'worship']
    if any(word in text_lower for word in religious_words):
        score += 1
    
    # Clear teaching indicators (+1)
    teaching_words = ['remember', 'learn', 'understand', 'know', 'lesson']
    if any(word in text_lower for word in teaching_words):
        score += 1
    
    # Important topics (+1)
    important_topics = ['Charity', 'Oppression', 'Dua', 'Mercy', 'Patience', 'Afterlife']
    if any(topic in important_topics):
        score += 1
    
    return min(score, 10)


def process_chunk_optimized(text, use_llm=True):
    """Process a single chunk with optimizations."""
    
    has_important = has_important_content(text)
    
    if not use_llm or not has_important:
        return {
            "summary": "Generic content - no summary needed",
            "topic": "Other",
            "confidence": 0.3,
            "clip_score": 3,
            "clip_candidate": False,
            "skipped_llm": True
        }
    
    # Optimized prompt (shorter = faster)
    prompt = f"""Analyze this Islamic lecture segment.

Text: {text[:600]}

Respond in JSON:
{{
  "summary": "1 sentence main lesson",
  "topic": "choose from: {', '.join(VALID_TOPICS[:8])}",
  "confidence": 0.0-1.0
}}

JSON:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.3,
            do_sample=True,
            top_p=0.9
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract JSON
    json_match = re.search(r'\{[^}]+\}', response, re.DOTALL)
    if json_match:
        try:
            result = json.loads(json_match.group())
            if result.get('topic') not in VALID_TOPICS:
                result['topic'] = 'Other'
            
            clip_score = calculate_clip_score(text, result.get('topic', ''), result.get('summary', ''))
            result['clip_score'] = clip_score
            result['clip_candidate'] = clip_score >= 7
            result['skipped_llm'] = False
            
            return result
        except:
            pass
    
    return {
        "summary": response.strip()[:150],
        "topic": "Other",
        "confidence": 0.5,
        "clip_score": 5,
        "clip_candidate": False,
        "skipped_llm": False
    }

## Step 6: Process with Checkpointing

In [ ]:
import time

# Check for resume
resume = False
start_chunk = 0
processed_chunks = []

checkpoints = gpu_mgr.list_checkpoints()
if checkpoints:
    latest_checkpoint = checkpoints[0]
    loaded_data, loaded_step = gpu_mgr.load_checkpoint(latest_checkpoint)
    
    if loaded_data:
        print(f"\nCheckpoint found: {latest_checkpoint.name}")
        print(f"Progress: {loaded_step}/{len(all_chunks)} chunks")
        
        # Manual resume prompt
        resume = gpu_mgr.show_resume_prompt(latest_checkpoint, loaded_step, len(all_chunks))
        
        if resume:
            processed_chunks = loaded_data
            start_chunk = loaded_step
            print(f"✓ Resuming from chunk {start_chunk}")
        else:
            print("✓ Starting fresh (ignoring checkpoint)")
    else:
        print("✓ No valid checkpoint - starting fresh")
else:
    print("✓ No checkpoints found - starting fresh")

# Processing configuration
BATCH_SIZE = gpu_mgr.get_batch_size()
CHECKPOINT_INTERVAL = 5  # Save checkpoint every 5 batches
total_chunks = len(all_chunks)

print(f"\nProcessing Configuration:")
print(f"  Total chunks: {total_chunks}")
print(f"  Starting from: {start_chunk}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Checkpoint interval: every {CHECKPOINT_INTERVAL} batches")
print()

# If resuming, skip already processed
chunks_to_process = all_chunks[start_chunk:] if resume else all_chunks

# Process in batches
skipped_count = 0
start_time = time.time()

for i in range(0, len(chunks_to_process), BATCH_SIZE):
    batch = chunks_to_process[i:i+BATCH_SIZE]
    batch_num = (i // BATCH_SIZE) + 1
    total_batches = (len(chunks_to_process) + BATCH_SIZE - 1) // BATCH_SIZE
    
    print(f"Processing batch {batch_num}/{total_batches}...")
    
    for chunk in batch:
        try:
            use_llm = has_important_content(chunk['text'])
            result = process_chunk_optimized(chunk['text'], use_llm=use_llm)
            
            chunk['summary'] = result['summary']
            chunk['topic'] = result['topic']
            chunk['confidence'] = result.get('confidence', 0.8)
            chunk['clip_score'] = result.get('clip_score', 5)
            chunk['clip_candidate'] = result.get('clip_candidate', False)
            chunk['skipped_llm'] = result.get('skipped_llm', False)
            
            if result['skipped_llm']:
                skipped_count += 1
                print(f"  ⏭️ Chunk {len(processed_chunks)+1}/{total_chunks} - Skipped (generic)")
            else:
                clip_flag = "🎬 " if result['clip_candidate'] else ""
                print(f"  ✓ Chunk {len(processed_chunks)+1}/{total_chunks} - {clip_flag}Topic: {result['topic']} (Score: {result['clip_score']})")
            
            processed_chunks.append(chunk)
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
            chunk['summary'] = "Processing error"
            chunk['topic'] = "Other"
            chunk['confidence'] = 0.0
            chunk['clip_score'] = 1
            chunk['clip_candidate'] = False
            chunk['skipped_llm'] = False
            processed_chunks.append(chunk)
    
    # Auto-checkpoint every N batches
    if batch_num % CHECKPOINT_INTERVAL == 0:
        current_step = start_chunk + len(processed_chunks)
        gpu_mgr.save_checkpoint(
            processed_chunks=processed_chunks,
            step=current_step,
            total_steps=total_chunks,
            metadata={
                "skipped_count": skipped_count,
                "batch_size": BATCH_SIZE,
                "gpu_type": gpu_type
            }
        )
        
        # Show stats
        elapsed = time.time() - start_time
        stats = gpu_mgr.get_processing_stats(len(processed_chunks), total_chunks, elapsed)
        print(f"  📊 Progress: {stats['progress']:.1f}% | ETA: {stats['eta_min']:.1f} min")

# Final stats
total_time = time.time() - start_time
print()
print(f"✓ Processing complete in {total_time/60:.1f} minutes")
print(f"✓ Processed {len(processed_chunks)} chunks")
print(f"✓ Skipped LLM for {skipped_count} chunks ({skipped_count/len(processed_chunks)*100:.1f}%)")
print(f"✓ Speed: {len(processed_chunks)/(total_time/60):.0f} chunks/minute")

## Step 7: Save & Download Results

In [ ]:
from collections import Counter

# Save final results
with open('processed_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(processed_chunks, f, indent=2, ensure_ascii=False)

print(f"✓ Saved to: processed_chunks.json")

# Statistics
topics = Counter([r['topic'] for r in processed_chunks])
clip_candidates = [r for r in processed_chunks if r.get('clip_candidate', False)]
high_scores = [r for r in processed_chunks if r.get('clip_score', 0) >= 8]

print(f"\n📊 Final Statistics:")
print(f"  Total chunks: {len(processed_chunks)}")
print(f"  Clip candidates (score ≥7): {len(clip_candidates)}")
print(f"  High priority clips (score ≥8): {len(high_scores)}")
print(f"\n📋 Topic distribution:")
for topic, count in topics.most_common():
    print(f"  {topic}: {count}")

In [ ]:
# Download results
from google.colab import files

print("\n📥 Downloading processed_chunks.json...")
files.download('processed_chunks.json')

# Save checkpoint one final time
gpu_mgr.save_checkpoint(
    processed_chunks=processed_chunks,
    step=len(processed_chunks),
    total_steps=len(processed_chunks),
    metadata={"status": "complete"}
)

print()
print("="*60)
print("NEXT STEP:")
print("="*60)
print("1. The file will download automatically")
print("2. On your computer, run:")
print("   python colab/import_results.py processed_chunks.json")
print("="*60)